# WSMTE — 03_finbert_inference.ipynb
**KAGGLE GPU (T4 x2)** | Day 1 Kaggle Tasks 1.11–1.20

Runs FinBERT (ProsusAI/finbert) and mDeBERTa (cross-encoder/nli-deberta-v3-small) on all three news datasets. Save each CSV immediately after each dataset.

**Kaggle input datasets required:**
- `/kaggle/input/wsmte-raw/kotekar_news.csv`
- `/kaggle/input/wsmte-raw/kaggle_news_1.csv`
- `/kaggle/input/wsmte-raw/kaggle_news_2.csv`

In [ ]:
# ── Cell 1: Clone repo + sys.path + CONFIG ─────────────────────────────────
!git clone https://github.com/IAMNEERAJ05/WSMTE.git
import sys
sys.path.insert(0, '/kaggle/working/WSMTE')

from config.config import CONFIG
print(f"FinBERT model  : {CONFIG['finbert_model']}")
print(f"mDeBERTa model : {CONFIG['mdeberta_model']}")
print(f"Date range     : {CONFIG['start_date']} to {CONFIG['end_date']}")

## GPU Check

In [ ]:
# ── Cell 2: Verify GPU ────────────────────────────────────────────────────
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch device : {device}')
if device == 'cuda':
    print(f'GPU name       : {torch.cuda.get_device_name(0)}')
    print(f'GPU count      : {torch.cuda.device_count()}')
else:
    print('WARNING: No GPU — enable T4 x2 in Kaggle Settings → Accelerator')

## Load FinBERT + mDeBERTa Models
> First run downloads from HuggingFace (~1–2 min each).

In [ ]:
# ── Cell 3: Load models ────────────────────────────────────────────────────
from src.sentiment.finbert_inference import load_finbert, load_mdeberta

print('Loading FinBERT...')
finbert_tokenizer, finbert_model, device = load_finbert(CONFIG, device=device)
print(f'FinBERT loaded on {device} OK')

print('\nLoading mDeBERTa...')
deberta_tokenizer, deberta_model, _ = load_mdeberta(CONFIG, device=device)
print(f'mDeBERTa loaded on {device} OK')

## Part 1 — Kotekar: FinBERT → `polarity_company`
> Input: `headline + first 2 sentences of articleBody`  
> Runtime: ~15–20 min

In [ ]:
# ── Cell 4: Load + filter Kotekar ─────────────────────────────────────────
import pandas as pd
from src.sentiment.finbert_inference import prepare_kotekar_text

kotekar = pd.read_csv('/kaggle/input/wsmte-raw/kotekar_news.csv')
kotekar['date'] = pd.to_datetime(kotekar[CONFIG['kotekar_date_col']]).dt.date
kotekar = kotekar[
    (kotekar['date'] >= pd.to_datetime(CONFIG['start_date']).date()) &
    (kotekar['date'] <= pd.to_datetime(CONFIG['end_date']).date())
].reset_index(drop=True)

print(f'Kotekar shape : {kotekar.shape}')
print(f'Date range    : {kotekar["date"].min()} to {kotekar["date"].max()}')
print(f'Columns       : {list(kotekar.columns)}')

In [ ]:
# ── Cell 5: Prepare text, run FinBERT ─────────────────────────────────────
from src.sentiment.finbert_inference import get_finbert_polarity

kotekar['text_for_finbert'] = kotekar.apply(prepare_kotekar_text, axis=1)
print('Sample text:', kotekar['text_for_finbert'].iloc[0][:120])
print(f'\nRunning FinBERT on {len(kotekar)} articles (~15-20 min)...')

kotekar['polarity_company'] = get_finbert_polarity(
    kotekar['text_for_finbert'].tolist(),
    tokenizer=finbert_tokenizer,
    model=finbert_model,
    device=device,
    batch_size=CONFIG['finbert_batch_size'],
    max_length=CONFIG['finbert_max_length'],
)
print('FinBERT done.')
print(f"polarity_company: min={kotekar['polarity_company'].min():.4f}  "
      f"max={kotekar['polarity_company'].max():.4f}  "
      f"mean={kotekar['polarity_company'].mean():.4f}")

# Checkpoint save — do NOT skip
kotekar[['date','company','symbol','polarity_company']].to_csv(
    '/kaggle/working/kotekar_polarity_temp.csv', index=False)
print('Checkpoint saved -> /kaggle/working/kotekar_polarity_temp.csv')

## Part 2 — Kotekar: mDeBERTa → `subjectivity`
> Input: full `articleBody` (truncated to 512 tokens)  
> Runtime: ~30–45 min

In [ ]:
# ── Cell 6: Run mDeBERTa subjectivity ─────────────────────────────────────
from src.sentiment.finbert_inference import get_subjectivity

print(f'Running mDeBERTa on {len(kotekar)} articles (~30-45 min)...')
print(f"Hypothesis: {CONFIG['subjectivity_hypothesis']}")

kotekar['subjectivity'] = get_subjectivity(
    kotekar[CONFIG['kotekar_text_col']].fillna('').tolist(),
    tokenizer=deberta_tokenizer,
    model=deberta_model,
    device=device,
    batch_size=CONFIG['mdeberta_batch_size'],
    max_length=CONFIG['mdeberta_max_length'],
)
print('mDeBERTa done.')
print(f"subjectivity: min={kotekar['subjectivity'].min():.4f}  "
      f"max={kotekar['subjectivity'].max():.4f}  "
      f"mean={kotekar['subjectivity'].mean():.4f}")

In [ ]:
# ── Cell 7: Save kotekar_sentiment.csv ─────────────────────────────────────
out = kotekar[['date', 'company', 'symbol', 'polarity_company', 'subjectivity']]
out.to_csv('/kaggle/working/kotekar_sentiment.csv', index=False)
print(f'Saved -> /kaggle/working/kotekar_sentiment.csv')
print(f'Shape : {out.shape}')
print(out.head(3).to_string())

## Part 3 — Kaggle Dataset 1: FinBERT → `polarity_market` (2020–2021)
> Text column: `Title`  
> Runtime: ~10–15 min

In [ ]:
# ── Cell 8: Load + filter Kaggle1 ─────────────────────────────────────────
kaggle1 = pd.read_csv('/kaggle/input/wsmte-raw/kaggle_news_1.csv')
kaggle1['date'] = pd.to_datetime(
    kaggle1[CONFIG['kaggle1_date_col']], format='%d/%m/%y').dt.date
kaggle1 = kaggle1[
    (kaggle1['date'] >= pd.to_datetime(CONFIG['start_date']).date()) &
    (kaggle1['date'] <= pd.to_datetime('2021-04-15').date())
].reset_index(drop=True)

print(f'Kaggle1 shape : {kaggle1.shape}')
print(f'Date range    : {kaggle1["date"].min()} to {kaggle1["date"].max()}')
print('Sample title  :', kaggle1[CONFIG['kaggle1_text_col']].iloc[0][:80])

In [ ]:
# ── Cell 9: Run FinBERT on Kaggle1 Title column ───────────────────────────
print(f'Running FinBERT on {len(kaggle1)} headlines (~10-15 min)...')

kaggle1['polarity_market'] = get_finbert_polarity(
    kaggle1[CONFIG['kaggle1_text_col']].fillna('').tolist(),
    tokenizer=finbert_tokenizer,
    model=finbert_model,
    device=device,
    batch_size=CONFIG['finbert_batch_size'],
    max_length=CONFIG['finbert_max_length'],
)
print('Kaggle1 FinBERT done.')
print(f"polarity_market: min={kaggle1['polarity_market'].min():.4f}  "
      f"max={kaggle1['polarity_market'].max():.4f}  "
      f"mean={kaggle1['polarity_market'].mean():.4f}")

kaggle1[['date', 'polarity_market']].to_csv(
    '/kaggle/working/kaggle1_polarity.csv', index=False)
print('Saved -> /kaggle/working/kaggle1_polarity.csv')

## Part 4 — Kaggle Dataset 2: FinBERT → `polarity_market` (2022–2024)
> Text column: `Headline`  
> Runtime: ~15–20 min

In [ ]:
# ── Cell 10: Load + filter Kaggle2 ────────────────────────────────────────
kaggle2 = pd.read_csv('/kaggle/input/wsmte-raw/kaggle_news_2.csv')
kaggle2['date'] = pd.to_datetime(
    kaggle2[CONFIG['kaggle2_date_col']], format='%d-%m-%Y').dt.date
kaggle2 = kaggle2[
    (kaggle2['date'] >= pd.to_datetime('2022-01-01').date()) &
    (kaggle2['date'] <= pd.to_datetime(CONFIG['end_date']).date())
].reset_index(drop=True)

print(f'Kaggle2 shape : {kaggle2.shape}')
print(f'Date range    : {kaggle2["date"].min()} to {kaggle2["date"].max()}')
print('Sample headline:', kaggle2[CONFIG['kaggle2_text_col']].iloc[0][:80])

In [ ]:
# ── Cell 11: Run FinBERT on Kaggle2 Headline column ──────────────────────
print(f'Running FinBERT on {len(kaggle2)} headlines (~15-20 min)...')

kaggle2['polarity_market'] = get_finbert_polarity(
    kaggle2[CONFIG['kaggle2_text_col']].fillna('').tolist(),
    tokenizer=finbert_tokenizer,
    model=finbert_model,
    device=device,
    batch_size=CONFIG['finbert_batch_size'],
    max_length=CONFIG['finbert_max_length'],
)
print('Kaggle2 FinBERT done.')
print(f"polarity_market: min={kaggle2['polarity_market'].min():.4f}  "
      f"max={kaggle2['polarity_market'].max():.4f}  "
      f"mean={kaggle2['polarity_market'].mean():.4f}")

kaggle2[['date', 'polarity_market']].to_csv(
    '/kaggle/working/kaggle2_polarity.csv', index=False)
print('Saved -> /kaggle/working/kaggle2_polarity.csv')

## Summary

In [ ]:
# ── Cell 12: Summary ─────────────────────────────────────────────────────
import os
outputs = [
    ('kotekar_sentiment.csv', '/kaggle/working/kotekar_sentiment.csv'),
    ('kaggle1_polarity.csv',  '/kaggle/working/kaggle1_polarity.csv'),
    ('kaggle2_polarity.csv',  '/kaggle/working/kaggle2_polarity.csv'),
]
print('=' * 55)
print('NOTEBOOK 03 COMPLETE')
print('=' * 55)
for name, path in outputs:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'  {name:30s}  shape={df.shape}')
    else:
        print(f'  {name:30s}  MISSING!')
print()
print('Download all 3 files from Kaggle Output tab to local data/finbert_outputs/')
print('Then run notebooks/01_data_prep.ipynb locally.')